# Obvious Failure Analysis

This notebook analyzes the 10-topic obvious-failure shortlist using the extracted bundle in `outputs/analysis_obvious_failure_bundle.json`.

It is designed to help answer:

- Which configurations fail for each topic?
- Does the error appear first in stage 1, stage 2, stage 3, or only at stage 4?
- Are there suspicious relation labels that may have corrupted the graph?
- Which topics look like graph failures versus final-judge overrides?


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 240)

REPO_ROOT = Path.cwd().resolve().parents[1]
BUNDLE_PATH = REPO_ROOT / 'outputs' / 'analysis_obvious_failure_bundle.json'
SHORTLIST_PATH = REPO_ROOT / 'outputs' / 'analysis_obvious_failure_shortlist.md'

bundle = json.loads(BUNDLE_PATH.read_text(encoding='utf-8'))
topics = bundle['topics']

print(f'Repo root: {REPO_ROOT}')
print(f'Loaded topics: {len(topics)}')
print(f'Bundle path: {BUNDLE_PATH}')

Repo root: C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate
Loaded topics: 10
Bundle path: C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\analysis_obvious_failure_bundle.json


In [2]:
CONFIG_ORDER = [
    'single_llm',
    'cot',
    'direct_judge',
    'two_agents',
    'six_agents',
    'targeted_attacks',
    'dung_no_agents',
    'full',
]


def build_frames(topics: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    topic_rows = []
    stage4_rows = []
    stage2_rows = []
    stage3_rows = []

    for topic in topics:
        topic_rows.append(
            {
                'topic_id': topic['topic_id'],
                'dataset': topic['dataset'],
                'domain': topic['domain'],
                'benchmark_label': topic['benchmark_label'],
                'topic_text': topic['topic_text'],
            }
        )

        for config, payload in topic['config_outputs'].items():
            stage4 = payload.get('stage4')
            if stage4:
                stage4_rows.append(
                    {
                        'topic_id': topic['topic_id'],
                        'dataset': topic['dataset'],
                        'domain': topic['domain'],
                        'topic_text': topic['topic_text'],
                        'config': config,
                        'verdict': stage4.get('verdict'),
                        'gold': stage4.get('benchmark_label'),
                        'confidence': stage4.get('confidence'),
                        'agree': stage4.get('verdict') == stage4.get('benchmark_label'),
                        'graph_verdict': stage4.get('graph_verdict'),
                        'used_graph': stage4.get('used_graph'),
                        'rationale': stage4.get('rationale'),
                    }
                )

            stage2 = payload.get('stage2')
            if stage2:
                summary = stage2.get('summary', {})
                counts = summary.get('label_counts', {})
                stage2_rows.append(
                    {
                        'topic_id': topic['topic_id'],
                        'dataset': topic['dataset'],
                        'config': config,
                        'kept_relations': summary.get('kept_relations'),
                        'failed_pairs': summary.get('failed_pairs', 0),
                        'avg_strength': summary.get('avg_strength'),
                        'attack_count': counts.get('Attack'),
                        'support_count': counts.get('Support'),
                        'neutral_count': counts.get('Neutral'),
                        'none_count': counts.get('None'),
                        'suspicious_relation_count': len(stage2.get('suspicious_relations', [])),
                    }
                )

            stage3 = payload.get('stage3')
            if stage3:
                verdict = stage3.get('graph_verdict', {})
                stage3_rows.append(
                    {
                        'topic_id': topic['topic_id'],
                        'dataset': topic['dataset'],
                        'config': config,
                        'graph_winner': verdict.get('winner'),
                        'graph_basis': verdict.get('basis'),
                        'pro_score': verdict.get('pro_score'),
                        'con_score': verdict.get('con_score'),
                        'margin': verdict.get('margin'),
                        'grounded_size': stage3.get('grounded_size'),
                        'n_attack_edges': stage3.get('n_attack_edges'),
                    }
                )

    topics_df = pd.DataFrame(topic_rows)
    stage4_df = pd.DataFrame(stage4_rows)
    stage2_df = pd.DataFrame(stage2_rows)
    stage3_df = pd.DataFrame(stage3_rows)

    if not stage4_df.empty:
        stage4_df['config'] = pd.Categorical(stage4_df['config'], CONFIG_ORDER, ordered=True)
        stage4_df = stage4_df.sort_values(['dataset', 'topic_id', 'config']).reset_index(drop=True)

    if not stage2_df.empty:
        stage2_df['config'] = pd.Categorical(stage2_df['config'], CONFIG_ORDER, ordered=True)
        stage2_df = stage2_df.sort_values(['dataset', 'topic_id', 'config']).reset_index(drop=True)

    if not stage3_df.empty:
        stage3_df['config'] = pd.Categorical(stage3_df['config'], CONFIG_ORDER, ordered=True)
        stage3_df = stage3_df.sort_values(['dataset', 'topic_id', 'config']).reset_index(drop=True)

    return topics_df, stage4_df, stage2_df, stage3_df


topics_df, stage4_df, stage2_df, stage3_df = build_frames(topics)

display(topics_df)
display(stage4_df.head(12))

,topic_id,dataset,domain,benchmark_label,topic_text
0,LOGIC_002,logic_test,syllogism,CON,"Resolved: If some birds can fly and penguins are birds, then penguins can fly."
1,LOGIC_005,logic_test,syllogism,CON,"Resolved: If it is raining then the ground is wet; the ground is wet, therefore it is raining."
2,LOGIC_030,logic_test,fallacy,CON,Resolved: The number of people who hold a belief is sufficient to determine whether that belief is true.
3,LOGIC_042,logic_test,empirical_fact,CON,Resolved: Humans only use 10 percent of their brains.
4,LOGIC_045,logic_test,empirical_fact,CON,Resolved: Antibiotics are effective against viral infections such as the common cold.
5,DDO_18636,ddo_sample,Science,CON,Evolution is not supported by science or evidence
6,DDO_50560,ddo_sample,Philosophy,CON,Science is a major threat to human existence
7,DDO_61182,ddo_sample,Technology,CON,Teenagers Should Have Unlimited Access to Computers and The Internet
8,DDO_70417,ddo_sample,Health,CON,This House believes that the United States Federal Government should add probiotics to all candy.
9,DDO_10643,ddo_sample,Education,CON,children should be allowed to box in school.


,topic_id,dataset,domain,topic_text,config,verdict,gold,confidence,agree,graph_verdict,used_graph,rationale
0,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,single_llm,CON,CON,0.80,True,None,False,Potential for severe injuries and lack of age-appropriate skill development in boxing.
1,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,cot,PRO,CON,0.90,False,None,False,"Proponents argue boxing teaches self-defense and discipline, outweighing the risks of injury."
2,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,direct_judge,PRO,CON,0.75,False,None,False,Pro arguments have stronger support and fewer attacks from the con side.
3,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,two_agents,PRO,CON,0.60,False,CON,True,PRO arguments have more undefeated claims and stronger backing from studies and safety measures.
4,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,six_agents,PRO,CON,0.75,False,None,False,Pro arguments have stronger evidence and fewer attacks from the con side.
5,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,targeted_attacks,PRO,CON,0.75,False,None,False,Pro arguments have stronger support and fewer attacks from the con side.
6,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,dung_no_agents,PRO,CON,0.85,False,PRO,True,"Children's motor skills and life skills are enhanced by boxing, outweighing potential risks."
7,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,full,PRO,CON,0.99,False,PRO,True,Logical structure and more undefeated claims for PRO.
8,DDO_18636,ddo_sample,Science,Evolution is not supported by science or evidence,single_llm,CON,CON,0.90,True,None,False,"Evolution is widely supported by extensive scientific evidence including fossil records, comparative anatomy, genetics, and experimental evolution studies."
9,DDO_18636,ddo_sample,Science,Evolution is not supported by science or evidence,cot,CON,CON,0.90,True,None,False,Evolution is widely supported by extensive scientific evidence including fossil records and genetic similarities.


## High-Level Overview

In [3]:
overview = (
    stage4_df.groupby(['dataset', 'config'], observed=True)
    .agg(
        n_topics=('topic_id', 'count'),
        n_correct=('agree', 'sum'),
        mean_confidence=('confidence', 'mean'),
    )
    .assign(acc=lambda df: df['n_correct'] / df['n_topics'])
    .reset_index()
)
display(overview)

pivot = overview.pivot(index='config', columns='dataset', values='acc').sort_index()
display(pivot.style.format('{:.1%}'))

,dataset,config,n_topics,n_correct,mean_confidence,acc
0,ddo_sample,single_llm,5,5,0.820,1.0
1,ddo_sample,cot,5,3,0.920,0.6
2,ddo_sample,direct_judge,5,0,0.850,0.0
3,ddo_sample,two_agents,5,0,0.780,0.0
4,ddo_sample,six_agents,5,0,0.850,0.0
5,ddo_sample,targeted_attacks,5,0,0.850,0.0
6,ddo_sample,dung_no_agents,5,0,0.880,0.0
7,ddo_sample,full,5,0,0.998,0.0
8,logic_test,single_llm,5,4,0.930,0.8
9,logic_test,cot,5,3,0.960,0.6


dataset,ddo_sample,logic_test
config,,
single_llm,100.0%,80.0%
cot,60.0%,60.0%
direct_judge,0.0%,20.0%
two_agents,0.0%,40.0%
six_agents,0.0%,0.0%
targeted_attacks,0.0%,0.0%
dung_no_agents,0.0%,40.0%
full,0.0%,0.0%


In [4]:
verdict_matrix = (
    stage4_df.assign(verdict_pair=stage4_df['verdict'] + '/' + stage4_df['gold'])
    .pivot_table(index=['dataset', 'topic_id', 'topic_text'], columns='config', values='verdict_pair', aggfunc='first')
    .reset_index()
)
display(verdict_matrix)

C:\Users\acer\AppData\Local\Temp\ipykernel_8696\4103984912.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index=['dataset', 'topic_id', 'topic_text'], columns='config', values='verdict_pair', aggfunc='first')


config,dataset,topic_id,topic_text,single_llm,cot,direct_judge,two_agents,six_agents,targeted_attacks,dung_no_agents,full
0,ddo_sample,DDO_10643,children should be allowed to box in school.,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
1,ddo_sample,DDO_18636,Evolution is not supported by science or evidence,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
2,ddo_sample,DDO_50560,Science is a major threat to human existence,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
3,ddo_sample,DDO_61182,Teenagers Should Have Unlimited Access to Computers and The Internet,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
4,ddo_sample,DDO_70417,This House believes that the United States Federal Government should add probiotics to all candy.,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
5,logic_test,LOGIC_002,"Resolved: If some birds can fly and penguins are birds, then penguins can fly.",CON/CON,CON/CON,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
6,logic_test,LOGIC_005,"Resolved: If it is raining then the ground is wet; the ground is wet, therefore it is raining.",PRO/CON,PRO/CON,PRO/CON,CON/CON,PRO/CON,PRO/CON,CON/CON,PRO/CON
7,logic_test,LOGIC_030,Resolved: The number of people who hold a belief is sufficient to determine whether that belief is true.,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
8,logic_test,LOGIC_042,Resolved: Humans only use 10 percent of their brains.,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON
9,logic_test,LOGIC_045,Resolved: Antibiotics are effective against viral infections such as the common cold.,CON/CON,CON/CON,PRO/CON,PRO/CON,PRO/CON,PRO/CON,CON/CON,PRO/CON


## Helper Functions

In [5]:
topic_index = {topic['topic_id']: topic for topic in topics}


def get_topic(topic_id: str) -> dict:
    return topic_index[topic_id]


def stage4_table(topic_id: str) -> pd.DataFrame:
    df = stage4_df[stage4_df['topic_id'] == topic_id].copy()
    return df[['config', 'verdict', 'gold', 'confidence', 'agree', 'graph_verdict', 'rationale']]


def stage2_table(topic_id: str) -> pd.DataFrame:
    df = stage2_df[stage2_df['topic_id'] == topic_id].copy()
    if df.empty:
        return df
    return df[['config', 'kept_relations', 'failed_pairs', 'attack_count', 'support_count', 'neutral_count', 'none_count', 'avg_strength', 'suspicious_relation_count']]


def stage3_table(topic_id: str) -> pd.DataFrame:
    df = stage3_df[stage3_df['topic_id'] == topic_id].copy()
    if df.empty:
        return df
    return df[['config', 'graph_winner', 'graph_basis', 'pro_score', 'con_score', 'margin', 'grounded_size', 'n_attack_edges']]


def show_stage1_samples(topic_id: str, config: str = 'full', n: int = 6) -> None:
    topic = get_topic(topic_id)
    payload = topic['config_outputs'][config]
    stage1 = payload.get('stage1')
    if not stage1:
        display(Markdown(f'No stage 1 data for `{topic_id}` / `{config}`'))
        return
    display(Markdown(f"### Stage 1 samples for `{topic_id}` / `{config}`"))
    args = pd.DataFrame(stage1['arguments'])
    display(args[['arg_id', 'stance', 'round', 'persona', 'targets_arg', 'text']].head(n))


def show_red_flags(topic_id: str, config: str = 'full', n: int = 10) -> None:
    topic = get_topic(topic_id)
    payload = topic['config_outputs'][config]
    stage2 = payload.get('stage2')
    if not stage2:
        display(Markdown(f'No stage 2 data for `{topic_id}` / `{config}`'))
        return
    red_flags = pd.DataFrame(stage2.get('suspicious_relations', []))
    display(Markdown(f"### Suspicious stage 2 relations for `{topic_id}` / `{config}`"))
    display(red_flags.head(n))


def show_grounded_arguments(topic_id: str, config: str = 'full', n: int = 12) -> None:
    topic = get_topic(topic_id)
    payload = topic['config_outputs'][config]
    stage3 = payload.get('stage3')
    if not stage3:
        display(Markdown(f'No stage 3 data for `{topic_id}` / `{config}`'))
        return
    grounded = pd.DataFrame(stage3.get('grounded_arguments', []))
    verdict = stage3.get('graph_verdict', {})
    display(Markdown(
        f"### Grounded extension for `{topic_id}` / `{config}`  \n"
        f"winner=`{verdict.get('winner')}`, basis=`{verdict.get('basis')}`, pro_score=`{verdict.get('pro_score')}`, con_score=`{verdict.get('con_score')}`"
    ))
    display(grounded.head(n))


def analyze_topic(topic_id: str) -> None:
    topic = get_topic(topic_id)
    display(Markdown(f"## {topic_id}"))
    display(Markdown(
        f"**Topic:** `{topic['topic_text']}`  \n"
        f"**Dataset:** `{topic['dataset']}`  \n"
        f"**Domain:** `{topic['domain']}`  \n"
        f"**Gold:** `{topic['benchmark_label']}`"
    ))
    display(stage4_table(topic_id))
    display(stage2_table(topic_id))
    display(stage3_table(topic_id))


## Error Pattern Checks

In [6]:
# Topics where the full graph winner matches gold, but full stage 4 is still wrong.
# These are strong candidates for a stage-4 override / judge problem.

full_stage4 = stage4_df[stage4_df['config'] == 'full'][['topic_id', 'dataset', 'gold', 'verdict', 'agree', 'confidence', 'rationale']]
full_stage3 = stage3_df[stage3_df['config'] == 'full'][['topic_id', 'graph_winner', 'graph_basis', 'pro_score', 'con_score', 'grounded_size']]

override_candidates = (
    full_stage4.merge(full_stage3, on='topic_id', how='left')
    .assign(graph_matches_gold=lambda df: df['graph_winner'] == df['gold'])
)
override_candidates = override_candidates[(override_candidates['agree'] == False) & (override_candidates['graph_matches_gold'])]
display(override_candidates.sort_values(['dataset', 'topic_id']))

,topic_id,dataset,gold,verdict,agree,confidence,rationale,graph_winner,graph_basis,pro_score,con_score,grounded_size,graph_matches_gold
5,LOGIC_002,logic_test,CON,PRO,False,1.00,PRO has more undefeated claims and stronger supporting attacks.,CON,grounded,0.0,1.0,7,True
8,LOGIC_042,logic_test,CON,PRO,False,0.95,Pro arguments are well-supported by neuroscientific evidence and refute the 10% myth.,CON,grounded,1.0,8.4,17,True
9,LOGIC_045,logic_test,CON,PRO,False,0.70,Pro arguments are more grounded and supported by scientific facts.,CON,grounded,3.0,5.0,13,True


In [7]:
# Topics where stage 2 has many suspicious relations in the full run.

relation_issues = (
    stage2_df[stage2_df['config'] == 'full']
    .merge(topics_df[['topic_id', 'topic_text', 'domain']], on='topic_id', how='left')
    .sort_values(['suspicious_relation_count', 'failed_pairs', 'kept_relations'], ascending=[False, False, False])
)
display(relation_issues[['topic_id', 'dataset', 'domain', 'suspicious_relation_count', 'failed_pairs', 'kept_relations']])

,topic_id,dataset,domain,suspicious_relation_count,failed_pairs,kept_relations
2,LOGIC_030,logic_test,fallacy,12,416,102
3,LOGIC_042,logic_test,empirical_fact,12,368,52
1,LOGIC_005,logic_test,syllogism,12,314,62
0,LOGIC_002,logic_test,syllogism,12,294,131
4,LOGIC_045,logic_test,empirical_fact,12,288,54


In [8]:
# Compare single-LLM versus full pipeline for the shortlist.

single = stage4_df[stage4_df['config'] == 'single_llm'][['topic_id', 'verdict', 'agree']].rename(columns={'verdict': 'single_verdict', 'agree': 'single_agree'})
full = stage4_df[stage4_df['config'] == 'full'][['topic_id', 'verdict', 'agree']].rename(columns={'verdict': 'full_verdict', 'agree': 'full_agree'})

drift = (
    topics_df[['topic_id', 'dataset', 'domain', 'topic_text', 'benchmark_label']]
    .merge(single, on='topic_id')
    .merge(full, on='topic_id')
    .assign(flipped=lambda df: df['single_verdict'] != df['full_verdict'])
)
display(drift.sort_values(['dataset', 'topic_id']))

,topic_id,dataset,domain,topic_text,benchmark_label,single_verdict,single_agree,full_verdict,full_agree,flipped
9,DDO_10643,ddo_sample,Education,children should be allowed to box in school.,CON,CON,True,PRO,False,True
5,DDO_18636,ddo_sample,Science,Evolution is not supported by science or evidence,CON,CON,True,PRO,False,True
6,DDO_50560,ddo_sample,Philosophy,Science is a major threat to human existence,CON,CON,True,PRO,False,True
7,DDO_61182,ddo_sample,Technology,Teenagers Should Have Unlimited Access to Computers and The Internet,CON,CON,True,PRO,False,True
8,DDO_70417,ddo_sample,Health,This House believes that the United States Federal Government should add probiotics to all candy.,CON,CON,True,PRO,False,True
0,LOGIC_002,logic_test,syllogism,"Resolved: If some birds can fly and penguins are birds, then penguins can fly.",CON,CON,True,PRO,False,True
1,LOGIC_005,logic_test,syllogism,"Resolved: If it is raining then the ground is wet; the ground is wet, therefore it is raining.",CON,PRO,False,PRO,False,False
2,LOGIC_030,logic_test,fallacy,Resolved: The number of people who hold a belief is sufficient to determine whether that belief is true.,CON,CON,True,PRO,False,True
3,LOGIC_042,logic_test,empirical_fact,Resolved: Humans only use 10 percent of their brains.,CON,CON,True,PRO,False,True
4,LOGIC_045,logic_test,empirical_fact,Resolved: Antibiotics are effective against viral infections such as the common cold.,CON,CON,True,PRO,False,True


## Topic Deep Dives

In [9]:
# Change this topic_id and rerun the next few cells while reading the markdown bundle.
topic_id = 'LOGIC_002'
analyze_topic(topic_id)

## LOGIC_002

**Topic:** `Resolved: If some birds can fly and penguins are birds, then penguins can fly.`  
**Dataset:** `logic_test`  
**Domain:** `syllogism`  
**Gold:** `CON`

,config,verdict,gold,confidence,agree,graph_verdict,rationale
40,single_llm,CON,CON,0.95,True,None,The resolution is false as it contradicts the biological fact that penguins cannot fly due to their physical adaptations for swimming rather than flying.
41,cot,CON,CON,1.00,True,None,"The definition of flight includes being able to leave the ground, which penguins cannot do due to their inability to walk upright and lack of wings suitable..."
42,direct_judge,CON,CON,0.85,True,None,CON has more undefeated claims and stronger attacks.
43,two_agents,CON,CON,0.80,True,TIE,CON has more undefeated claims and stronger attacks.
44,six_agents,PRO,CON,0.95,False,None,PRO arguments logically refute the resolution while CON arguments focus on specific biological adaptations.
45,targeted_attacks,PRO,CON,0.95,False,None,PRO arguments logically refute the resolution while CON arguments focus on specific biological adaptations.
46,dung_no_agents,PRO,CON,0.95,False,PRO,PRO has more undefeated claims and stronger supporting attacks.
47,full,PRO,CON,1.00,False,CON,PRO has more undefeated claims and stronger supporting attacks.


,config,kept_relations,failed_pairs,attack_count,support_count,neutral_count,none_count,avg_strength,suspicious_relation_count
20,direct_judge,324,1,90,217,138,107,0.333,12
21,two_agents,7,41,3,4,2,47,0.625,2
22,targeted_attacks,131,294,59,70,27,396,0.333,12
23,dung_no_agents,34,0,11,23,15,7,0.625,12
24,full,131,294,59,70,27,396,0.333,12


,config,graph_winner,graph_basis,pro_score,con_score,margin,grounded_size,n_attack_edges
15,two_agents,TIE,grounded,2.0,2.0,0.0,6,3
16,dung_no_agents,PRO,grounded,2.0,0.0,2.0,4,11
17,full,CON,grounded,0.0,1.0,-1.0,7,59


In [10]:
show_stage1_samples(topic_id, config='full', n=12)

### Stage 1 samples for `LOGIC_002` / `full`

,arg_id,stance,round,persona,targets_arg,text
0,LOGIC_002_A000,PRO,1,Rationalist Pro,NaN,Flying ability is not a universal trait among all bird species.
1,LOGIC_002_A001,PRO,1,Rationalist Pro,NaN,"While penguins belong to the avian class, they evolved flightlessness due to their aquatic lifestyle."
2,LOGIC_002_A002,PRO,1,Rationalist Pro,NaN,Quantitative analysis shows that only approximately 6% of bird species can fly; penguins fall outside this category.
3,LOGIC_002_A003,PRO,1,Ethics Advocate Pro,NaN,"According to ethical norms, all members of a species should have equal rights to flight."
4,LOGIC_002_A004,PRO,1,Ethics Advocate Pro,NaN,Penguins' inability to fly despite being classified as birds challenges our ethical understanding of these creatures.
5,LOGIC_002_A005,PRO,1,Ethics Advocate Pro,NaN,"The principle of equal treatment dictates that penguins, as birds, should not be denied the ability to fly based on their current condition."
6,LOGIC_002_A006,PRO,1,Futurist Pro,NaN,Flying ability is not solely determined by biological classification.
7,LOGIC_002_A007,PRO,1,Futurist Pro,NaN,"While penguins can fly in a metaphorical sense, their physical adaptations limit actual flight capability."
8,LOGIC_002_A008,PRO,1,Futurist Pro,NaN,Evolving penguin flight capabilities could unlock new economic opportunities in marine transportation.
9,LOGIC_002_A009,CON,1,Skeptic Con,NaN,"Some birds, including penguins, cannot fly."


In [11]:
show_red_flags(topic_id, config='full', n=12)

### Suspicious stage 2 relations for `LOGIC_002` / `full`

,source_arg_id,target_arg_id,source_stance,target_stance,label,confidence,premise,justification
0,LOGIC_002_A005,LOGIC_002_A012,PRO,CON,Support,1.0,Penguins' inability to fly is a fundamental right,None
1,LOGIC_002_A005,LOGIC_002_A013,PRO,CON,Support,1.0,Fundamental right to fulfill natural behaviors,None
2,LOGIC_002_A005,LOGIC_002_A014,PRO,CON,Support,1.0,Natural behavior as defined by species,None
3,LOGIC_002_A006,LOGIC_002_A011,PRO,CON,Support,1.0,Flying ability is not solely determined by biological classification.,None
4,LOGIC_002_A006,LOGIC_002_A013,PRO,CON,Support,1.0,Flying ability is not solely determined by biological classification.,None
5,LOGIC_002_A008,LOGIC_002_A012,PRO,CON,Support,1.0,fundamental right to natural behavior,None
6,LOGIC_002_A008,LOGIC_002_A013,PRO,CON,Support,1.0,ethical norm,None
7,LOGIC_002_A009,LOGIC_002_A021,CON,PRO,Support,1.0,"Penguin wings are perfectly adapted for swimming, not flying, aligning with their aquatic lifestyle.",None
8,LOGIC_002_A010,LOGIC_002_A018,CON,PRO,Support,1.0,"Some birds, including penguins, cannot fly.",None
9,LOGIC_002_A012,LOGIC_002_A003,CON,PRO,Support,1.0,they evolved flightlessness due to their aquatic lifestyle,None


In [12]:
show_grounded_arguments(topic_id, config='full', n=12)

### Grounded extension for `LOGIC_002` / `full`  
winner=`CON`, basis=`grounded`, pro_score=`0.0`, con_score=`1.0`

,arg_id,stance,round,persona,text
0,LOGIC_002_A000,PRO,1,Rationalist Pro,Flying ability is not a universal trait among all bird species.
1,LOGIC_002_A006,PRO,1,Futurist Pro,Flying ability is not solely determined by biological classification.
2,LOGIC_002_A008,PRO,1,Futurist Pro,Evolving penguin flight capabilities could unlock new economic opportunities in marine transportation.
3,LOGIC_002_A010,CON,1,Skeptic Con,Logical fallacy of affirming the consequent is present in the resolution's structure.
4,LOGIC_002_A013,CON,1,Rights Defender Con,The resolution's assertion overlooks the ethical norm that all creatures have the right to exist without being subject to unnatural conditions that prevent ...
5,LOGIC_002_A020,PRO,2,Ethics Advocate Pro,"While some birds can't fly, penguins, as birds, are included in this category."
6,LOGIC_002_A023,PRO,2,Futurist Pro,"Affirming the consequent isn't applicable here; it's about biological facts, not logic structure."


## Suggested Workflow

For each shortlisted topic:

1. Start with the `stage4_table(topic_id)` output and mark whether the error is widespread or only appears in later configurations.
2. Check `stage3_table(topic_id)` to see whether the graph already prefers the wrong side.
3. If the graph is wrong, inspect `show_red_flags(...)` and `show_grounded_arguments(...)`.
4. If the graph is correct but stage 4 is wrong, classify it as a likely stage-4 override / final-judge failure.
5. Use `show_stage1_samples(...)` to determine whether the raw arguments were already contaminated by weak or contradictory content.
